# **Final Project - Ensemble Models**

- Bryan Keating
- Python and Math for Machine Learning
- 14 December 2025

## Task 1 - Logistic Regression Model Implementation

In [17]:
# Dependencies and libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [18]:
# Load dataset
df = pd.read_csv("allwine.csv")
df.head()

,Unnamed: 0,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,density,pH,sulphates,alcohol,quality
0,0,-0.743787,0.805266,-1.455948,-0.541531,-0.334525,-0.539436,-0.978159,0.146723,-0.755850,-1.297136,0
1,1,-0.520914,1.798500,-1.455948,-0.047918,0.129345,0.787432,-0.998211,-1.220838,-0.062351,-0.960761,0
2,2,-0.520914,1.136344,-1.251203,-0.259467,0.002835,-0.160331,-0.994200,-0.956148,-0.235726,-0.960761,0
3,3,1.373509,-1.512280,1.410480,-0.541531,-0.355610,0.029222,-0.974148,-1.397297,-0.640267,-0.960761,1
4,4,-0.743787,0.805266,-1.455948,-0.541531,-0.334525,-0.539436,-0.978159,0.146723,-0.755850,-1.297136,0


In [19]:
# Check for missing values
df.isnull().sum()

Unnamed: 0             0
fixed acidity          0
volatile acidity       0
citric acid            0
residual sugar         0
chlorides              0
free sulfur dioxide    0
density                0
pH                     0
sulphates              0
alcohol                0
quality                0
dtype: int64

In [20]:
# Split features and target
inp_df = df.drop("quality", axis=1)
out_df = df["quality"]
out_df.head()

0    0
1    0
2    0
3    1
4    0
Name: quality, dtype: int64

In [21]:
# Scale features (even though they are already nomalized)
scaler = StandardScaler()
inp_df = scaler.fit_transform(inp_df)

In [22]:
# Extracting train and test sets
X_train, X_test, y_train, y_test = train_test_split(inp_df, out_df, test_size=0.2, random_state=42)

In [23]:
# Rename and reformat
X_tr_arr = X_train
X_ts_arr = X_test
y_tr_arr = y_train.to_numpy()
y_ts_arr = y_test.to_numpy()

In [24]:
# View data
print('Input Shape:', X_tr_arr.shape)
print('Output Shape:', y_tr_arr.shape)

Input Shape: (2558, 11)
Output Shape: (2558,)


In [25]:
def weightInitialization(n_features):
    w = np.zeros((1, n_features))
    b = 0
    return w, b

In [26]:
def sigmoid_activation(result):
    final_result = 1 / (1 + np.exp(-result))
    return final_result


In [27]:
def model_optimize(w, b, X, Y):
    m = X.shape[0]
    
    # Prediction
    final_result = sigmoid_activation(np.dot(w, X.T) + b)
    Y_T = Y.T
    cost = (-1/m) * (
        np.sum((Y_T * np.log(final_result)) +
               ((1 - Y_T) * (np.log(1 - final_result))))
    )
    
    # Gradient calculation
    dw = (1/m) * (np.dot(X.T, (final_result - Y.T).T))
    db = (1/m) * (np.sum(final_result - Y.T))
    
    grads = {"dw": dw, "db": db}
    
    return grads, cost


In [28]:
def model_predict(w, b, X, Y, learning_rate, no_iterations):
    costs = []
    for i in range(no_iterations):
        grads, cost = model_optimize(w, b, X, Y)
        
        dw = grads["dw"]
        db = grads["db"]
        
        # Weight update (gradient descent step)
        w = w - (learning_rate * dw.T)
        b = b - (learning_rate * db)
        
        if (i % 100 == 0):
            costs.append(cost)
    
    coeff = {"w": w, "b": b}
    gradient = {"dw": dw, "db": db}
    
    return coeff, gradient, costs


In [29]:
def predict(final_pred, m):
    y_pred = np.zeros((1,m))
    for i in range(final_pred.shape[1]):
        if final_pred[0][i] > 0.5:
            y_pred[0][i] = 1
    return y_pred

In [30]:
# Determine model size
n_features = X_tr_arr.shape[1]
print('Number of features:', n_features)

# Initialize model weights
w, b = weightInitialization(n_features)

# Train the model using gradient descent
coeff, gradient, costs = model_predict(
    w, b,
    X_tr_arr, y_tr_arr,
    learning_rate=0.0001,
    no_iterations=4500
)

# Extract trained weights
w = coeff["w"]
b = coeff["b"]
print('Optimized weights', w)
print('Optimized intercept',b)

# Generate probabilities
final_train_pred = sigmoid_activation(np.dot(w, X_tr_arr.T) + b)
final_test_pred = sigmoid_activation(np.dot(w, X_ts_arr.T) + b)

# Convert proabilities to binary prediction
m_tr = X_tr_arr.shape[0]
m_ts = X_ts_arr.shape[0]
y_tr_pred = predict(final_train_pred, m_tr)
y_ts_pred = predict(final_test_pred, m_ts)

# Evaluate model performance
print('Training Accuracy', accuracy_score(y_tr_pred.T, y_tr_arr))
print('Test Accuracy', accuracy_score(y_ts_pred.T, y_ts_arr))

Number of features: 11
Optimized weights [[ 9.18876626e-03  1.77197099e-02 -6.56797996e-02  3.10443617e-02
  -1.19610572e-03 -2.37243168e-02 -1.43054880e-02 -2.13805933e-03
   9.93343389e-05  4.44673413e-02  8.12137352e-02]]
Optimized intercept 0.018816799910001045
Training Accuracy 0.7044566067240031
Test Accuracy 0.6890625


## Task 2 - Dynamic Ensemble Logistic Regression Model

### LR_middle Model Training

In [31]:
# Determine model size
n_features = X_tr_arr.shape[1]
print('Number of features:', n_features)

# Initialize LR_middle weights
w_M, b_M = weightInitialization(n_features)

# Train LR_middle using gradient descent
coeff_M, gradient_M, costs_M = model_predict(
    w_M, b_M,
    X_tr_arr, y_tr_arr,
    learning_rate=0.0001,
    no_iterations=4500
)

# Extract trained LR_middle weights
w_M = coeff_M["w"]
b_M = coeff_M["b"]
print('Optimized weights (Middle):', w_M)
print('Optimized intercept (Middle):', b_M)

# Generate LR_middle probabilities
h_M_train = sigmoid_activation(np.dot(w_M, X_tr_arr.T) + b_M)
h_M_test  = sigmoid_activation(np.dot(w_M, X_ts_arr.T) + b_M)

# Convert probabilities to binary prediction (sanity check)
m_tr = X_tr_arr.shape[0]
m_ts = X_ts_arr.shape[0]
y_M_tr_pred = predict(h_M_train, m_tr)
y_M_ts_pred = predict(h_M_test, m_ts)

# Evaluate LR_middle performance
print('LR_middle Training Accuracy:', accuracy_score(y_M_tr_pred.T, y_tr_arr))
print('LR_middle Test Accuracy:', accuracy_score(y_M_ts_pred.T, y_ts_arr))

Number of features: 11
Optimized weights (Middle): [[ 9.18876626e-03  1.77197099e-02 -6.56797996e-02  3.10443617e-02
  -1.19610572e-03 -2.37243168e-02 -1.43054880e-02 -2.13805933e-03
   9.93343389e-05  4.44673413e-02  8.12137352e-02]]
Optimized intercept (Middle): 0.018816799910001045
LR_middle Training Accuracy: 0.7044566067240031
LR_middle Test Accuracy: 0.6890625


### Slit Training Data

In [ ]:
# Get middle model probabilities on training data
h_M_train = sigmoid_activation(np.dot(w_M, X_tr_arr.T) + b_M).flatten()

# Split based on gating probability
left_idx  = h_M_train >= 0.5
right_idx = h_M_train < 0.5

# Create LEFT training set
X_left = X_tr_arr[left_idx]
y_left = y_tr_arr[left_idx]

# Create RIGHT training set
X_right = X_tr_arr[right_idx]
y_right = y_tr_arr[right_idx]

### LR_left Model Training

In [ ]:
# Determine model size
n_features = X_tr_arr.shape[1]
print('Number of features:', n_features)

# Initialize LR_left weights
w_L, b_L = weightInitialization(X_left.shape[1])

# Train LR_left using gradient descent
coeff_L, gradient_L, costs_L = model_predict(
    w_L, b_L,
    X_tr_arr, y_tr_arr,
    learning_rate=0.0001,
    no_iterations=4500
)

# Extract trained LR_left weights
w_L = coeff_L["w"]
b_L = coeff_L["b"]
print('Optimized weights (Left):', w_L)
print('Optimized intercept (Left):', b_L)

# Generate LR_left probabilities
h_L_train = sigmoid_activation(np.dot(w_L, X_tr_arr.T) + b_L)
h_L_test  = sigmoid_activation(np.dot(w_L, X_ts_arr.T) + b_L)

# Convert probabilities to binary prediction (sanity check)
m_tr = X_tr_arr.shape[0]
m_ts = X_ts_arr.shape[0]
y_L_tr_pred = predict(h_L_train, m_tr)
y_L_ts_pred = predict(h_L_test, m_ts)

# Evaluate LR_left performance
print('LR_left Training Accuracy:', accuracy_score(y_L_tr_pred.T, y_tr_arr))
print('LR_left Test Accuracy:', accuracy_score(y_L_ts_pred.T, y_ts_arr))

Number of features: 11
Optimized weights (Left): [[ 9.18876626e-03  1.77197099e-02 -6.56797996e-02  3.10443617e-02
  -1.19610572e-03 -2.37243168e-02 -1.43054880e-02 -2.13805933e-03
   9.93343389e-05  4.44673413e-02  8.12137352e-02]]
Optimized intercept (Left): 0.018816799910001045
LR_left Training Accuracy: 0.7044566067240031
LR_left Test Accuracy: 0.6890625


### LR_right Model Training

In [33]:
# Determine model size
n_features = X_tr_arr.shape[1]
print('Number of features:', n_features)

# Initialize LR_right weights
w_R, b_R = weightInitialization(n_features)

# Train LR_right using gradient descent
coeff_R, gradient_R, costs_R = model_predict(
    w_R, b_R,
    X_tr_arr, y_tr_arr,
    learning_rate=0.0001,
    no_iterations=4500
)

# Extract trained LR_right weights
w_R = coeff_R["w"]
b_R = coeff_R["b"]
print('Optimized weights (Right):', w_R)
print('Optimized intercept (Right):', b_R)

# Generate LR_right probabilities
h_R_train = sigmoid_activation(np.dot(w_R, X_tr_arr.T) + b_R)
h_R_test  = sigmoid_activation(np.dot(w_R, X_ts_arr.T) + b_R)

# Convert probabilities to binary prediction (sanity check)
m_tr = X_tr_arr.shape[0]
m_ts = X_ts_arr.shape[0]
y_R_tr_pred = predict(h_R_train, m_tr)
y_R_ts_pred = predict(h_R_test, m_ts)

# Evaluate LR_right performance
print('LR_right Training Accuracy:', accuracy_score(y_R_tr_pred.T, y_tr_arr))
print('LR_right Test Accuracy:', accuracy_score(y_R_ts_pred.T, y_ts_arr))

Number of features: 11
Optimized weights (Right): [[ 9.18876626e-03  1.77197099e-02 -6.56797996e-02  3.10443617e-02
  -1.19610572e-03 -2.37243168e-02 -1.43054880e-02 -2.13805933e-03
   9.93343389e-05  4.44673413e-02  8.12137352e-02]]
Optimized intercept (Right): 0.018816799910001045
LR_right Training Accuracy: 0.7044566067240031
LR_right Test Accuracy: 0.6890625


### Ensemble Model

In [34]:
# Combine probabilities using the ensemble formula
ensemble_prob = (h_L_train * h_M_train) + (h_R_train * (1 - h_M_train))

# Convert combined probabilities to binary predictions
y_ensemble_tr_pred = predict(ensemble_prob, m_tr)

# Repeat for the test set
ensemble_prob_test = (h_L_test * h_M_test) + (h_R_test * (1 - h_M_test))
y_ensemble_ts_pred = predict(ensemble_prob_test, m_ts)

# Evaluate the ensemble model's performance
print('Ensemble Training Accuracy:', accuracy_score(y_ensemble_tr_pred.T, y_tr_arr))
print('Ensemble Test Accuracy:', accuracy_score(y_ensemble_ts_pred.T, y_ts_arr))

Ensemble Training Accuracy: 0.7044566067240031
Ensemble Test Accuracy: 0.6890625
